# 스트레스 점수 예측 V21 — Precision-Fixed 2D/3D/4D Neighbor

**Repeated-CV Higher-Order Neighbor Ensemble**

- Public 0.1272333 원본의 float32 순위 계산을 정확히 재현
- 8개 핵심 변수의 70개 4D 조합에서 Train 1-NN 추가
- 2D·3D·4D 분위수와 혼합 비율을 3개 5-Fold 분할에서 비교
- 3개 분할 모두 정밀도를 수정한 V19보다 0.0001 이상 좋은 4D 조합만 선택
- 모든 순위·이웃 탐색은 Fold Train으로만 학습

In [9]:
from pathlib import Path
import os
from itertools import combinations

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 1. 설정

In [10]:
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")
TARGET = "stress_score"
ID_COLUMN = "ID"
N_ESTIMATORS = 1200
MODEL_SEED = 42
CV_SEEDS = (42, 77, 2026)
TREE_QUANTILE = 0.52

PAIR_QUANTILES = (0.48, 0.50, 0.52)
TRIPLE_QUANTILES = (0.46, 0.48, 0.50)
QUAD_QUANTILES = (0.46, 0.48, 0.50, 0.52)
PAIR_WEIGHTS = (0.05, 0.10, 0.15)
TRIPLE_WEIGHTS = (0.05, 0.10, 0.15)
QUAD_WEIGHTS = (0.00, 0.05, 0.10, 0.15)
ROUND_OPTIONS = (False, True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2,
    "height": 2, "glucose": 3, "weight": 2,
    "cholesterol_glucose_ratio": 2, "bone_density": 4,
}
NEIGHBOR_COLUMNS = list(FEATURE_COPY_COUNTS)

## 2. 데이터와 파생변수

In [11]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("ID 순서가 다릅니다.")

def add_features(frame):
    r=frame.copy(); hm=r["height"]/100
    sbp=r["systolic_blood_pressure"]; dbp=r["diastolic_blood_pressure"]
    r["bmi"]=r["weight"]/hm.pow(2)
    r["pulse_pressure"]=sbp-dbp
    r["mean_arterial_pressure"]=(sbp+2*dbp)/3
    r["blood_pressure_ratio"]=sbp/(dbp+1e-6)
    r["cholesterol_glucose_ratio"]=r["cholesterol"]/(r["glucose"]+1e-6)
    r["missing_count"]=r.isna().sum(axis=1)
    return r

def add_copies(frame):
    r=frame.copy()
    for f,n in FEATURE_COPY_COUNTS.items():
        for i in range(1,n+1): r[f"{f}__copy{i}"]=r[f]
    return r

y=train[TARGET].reset_index(drop=True)
X=add_features(train.drop(columns=[TARGET,ID_COLUMN])).reset_index(drop=True)
X_test=add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
print("Train:",X.shape,"| 2D + 3D + 4D combinations:",len(list(combinations(NEIGHBOR_COLUMNS,2))), "+", len(list(combinations(NEIGHBOR_COLUMNS,3))), "+", len(list(combinations(NEIGHBOR_COLUMNS,4))))

Train: (3000, 22) | 2D + 3D + 4D combinations: 28 + 56 + 70


## 3. ExtraTrees와 Train-기준 순위 함수

In [12]:
def make_preprocessor(frame):
    cat=frame.select_dtypes(include=["object","category","string"]).columns.tolist()
    num=frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("cat",Pipeline([
            ("imp",SimpleImputer(strategy="constant",fill_value="missing")),
            ("oh",OneHotEncoder(handle_unknown="ignore",sparse_output=False)),
        ]),cat),
        ("num",SimpleImputer(strategy="median",add_indicator=True),num),
    ])

def tree_predict(train_x, valid_x, target):
    wt,wv=add_copies(train_x),add_copies(valid_x)
    p=make_preprocessor(wt); a=p.fit_transform(wt); b=p.transform(wv)
    m=ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,min_samples_leaf=1,max_features=1,
        random_state=MODEL_SEED,n_jobs=-1,
    )
    m.fit(a,target)
    matrix=np.column_stack([t.predict(b) for t in m.estimators_])
    return np.clip(np.round(np.quantile(matrix,TREE_QUANTILE,axis=1),2),0,1)

def rank_transform(train_values, valid_values):
    median=np.nanmedian(train_values)
    a=np.nan_to_num(train_values,nan=median)
    b=np.nan_to_num(valid_values,nan=median)
    sorted_a=np.sort(a)
    train_rank=np.searchsorted(sorted_a,a,side="right")/len(a)
    valid_rank=np.searchsorted(sorted_a,b,side="right")/len(a)
    return train_rank.astype(np.float32),valid_rank.astype(np.float32)

## 4. 2D·3D·4D 1-NN 예측 소재 생성

In [13]:
def neighbor_materials(train_x, valid_x, target):
    train_rank={}; valid_rank={}
    for col in NEIGHBOR_COLUMNS:
        train_rank[col],valid_rank[col]=rank_transform(
            train_x[col].to_numpy(float),valid_x[col].to_numpy(float)
        )
    target_values=target.to_numpy(float)
    output={2:[],3:[],4:[]}
    for dimension in (2,3,4):
        for columns in combinations(NEIGHBOR_COLUMNS,dimension):
            distance=np.zeros((len(valid_x),len(train_x)),dtype=np.float32)
            for column in columns:
                distance += np.abs(
                    valid_rank[column][:,None]-train_rank[column][None,:]
                ).astype(np.float32)
            # 기준 V7과 동일한 np.argmin 동률 처리
            nearest=np.argmin(distance,axis=1)
            output[dimension].append(target_values[nearest])
    return {dimension:np.column_stack(values) for dimension,values in output.items()}

## 5. 3개 CV 분할에서 float32 2D·3D·4D OOF 생성

In [14]:
split_data={}
quantiles_by_dimension={
    2:PAIR_QUANTILES,3:TRIPLE_QUANTILES,4:QUAD_QUANTILES
}
for cv_seed in CV_SEEDS:
    cv=KFold(5,shuffle=True,random_state=cv_seed)
    tree_oof=np.zeros(len(y))
    neighbor_oof={
        (dimension,quantile):np.zeros(len(y))
        for dimension,quantiles in quantiles_by_dimension.items()
        for quantile in quantiles
    }
    for fold,(ti,vi) in enumerate(cv.split(X),1):
        tree_oof[vi]=tree_predict(X.iloc[ti],X.iloc[vi],y.iloc[ti])
        materials=neighbor_materials(X.iloc[ti],X.iloc[vi],y.iloc[ti])
        for dimension,quantiles in quantiles_by_dimension.items():
            for quantile in quantiles:
                neighbor_oof[(dimension,quantile)][vi]=np.quantile(
                    materials[dimension],quantile,axis=1
                )
        print(f"CV {cv_seed} | Fold {fold}: complete")
    split_data[cv_seed]={"tree":tree_oof,"neighbor":neighbor_oof}
    baseline=np.clip(np.round(
        .8*tree_oof+.1*neighbor_oof[(2,.50)]+.1*neighbor_oof[(3,.48)],2
    ),0,1)
    print(f"CV {cv_seed} float32 V19 baseline MAE: {mean_absolute_error(y,baseline):.6f}")

CV 42 | Fold 1: complete
CV 42 | Fold 2: complete
CV 42 | Fold 3: complete
CV 42 | Fold 4: complete
CV 42 | Fold 5: complete
CV 42 float32 V19 baseline MAE: 0.148320
CV 77 | Fold 1: complete
CV 77 | Fold 2: complete
CV 77 | Fold 3: complete
CV 77 | Fold 4: complete
CV 77 | Fold 5: complete
CV 77 float32 V19 baseline MAE: 0.146227
CV 2026 | Fold 1: complete
CV 2026 | Fold 2: complete
CV 2026 | Fold 3: complete
CV 2026 | Fold 4: complete
CV 2026 | Fold 5: complete
CV 2026 float32 V19 baseline MAE: 0.144610


## 6. 모든 분할에서 안정적인 4D 조합 선택

In [15]:
rows=[]
for pair_q in PAIR_QUANTILES:
 for triple_q in TRIPLE_QUANTILES:
  for quad_q in QUAD_QUANTILES:
   for pair_weight in PAIR_WEIGHTS:
    for triple_weight in TRIPLE_WEIGHTS:
     for quad_weight in QUAD_WEIGHTS:
      total_neighbor=pair_weight+triple_weight+quad_weight
      if total_neighbor>=.5: continue
      for rounding in ROUND_OPTIONS:
       scores=[]; improvements=[]
       for cv_seed in CV_SEEDS:
        item=split_data[cv_seed]
        baseline=np.clip(np.round(
            .8*item["tree"]+.1*item["neighbor"][(2,.50)]
            +.1*item["neighbor"][(3,.48)],2
        ),0,1)
        pred=(
            (1-total_neighbor)*item["tree"]
            +pair_weight*item["neighbor"][(2,pair_q)]
            +triple_weight*item["neighbor"][(3,triple_q)]
            +quad_weight*item["neighbor"][(4,quad_q)]
        )
        if rounding: pred=np.round(pred,2)
        pred=np.clip(pred,0,1)
        base_mae=mean_absolute_error(y,baseline); score=mean_absolute_error(y,pred)
        scores.append(score); improvements.append(base_mae-score)
       rows.append({
        "pair_quantile":pair_q,"triple_quantile":triple_q,"quad_quantile":quad_q,
        "pair_weight":pair_weight,"triple_weight":triple_weight,"quad_weight":quad_weight,
        "tree_weight":1-total_neighbor,"rounding":rounding,
        "mean_CV_MAE":np.mean(scores),"mean_improvement":np.mean(improvements),
        "worst_improvement":min(improvements),"wins":sum(v>0 for v in improvements),
       })
results=pd.DataFrame(rows).sort_values(
    ["wins","worst_improvement","mean_CV_MAE"],ascending=[False,False,True]
).reset_index(drop=True)
eligible=results[
    (results.wins==len(CV_SEEDS))&(results.worst_improvement>=.0001)&(results.quad_weight>0)
]
if len(eligible): selected=eligible.iloc[0]
else:
    selected=results[
        (results.pair_quantile==.50)&(results.triple_quantile==.48)&
        (results.pair_weight==.10)&(results.triple_weight==.10)&
        (results.quad_weight==0)&(results.rounding)
    ].iloc[0]
print("Selected tree weight:",selected.tree_weight)
print("Selected 2D q / weight:",selected.pair_quantile,"/",selected.pair_weight)
print("Selected 3D q / weight:",selected.triple_quantile,"/",selected.triple_weight)
print("Selected 4D q / weight:",selected.quad_quantile,"/",selected.quad_weight)
print("Round to 2 decimals:",bool(selected.rounding))
print("Repeated-CV mean MAE:",f"{selected.mean_CV_MAE:.6f}")
print("Worst split improvement:",f"{selected.worst_improvement:+.6f}")
print("Wins:",int(selected.wins),"/",len(CV_SEEDS))
results.head(20)

Selected tree weight: 0.8
Selected 2D q / weight: 0.5 / 0.1
Selected 3D q / weight: 0.48 / 0.1
Selected 4D q / weight: 0.46 / 0.0
Round to 2 decimals: True
Repeated-CV mean MAE: 0.146386
Worst split improvement: +0.000000
Wins: 0 / 3


,pair_quantile,triple_quantile,quad_quantile,pair_weight,triple_weight,quad_weight,tree_weight,rounding,mean_CV_MAE,mean_improvement,worst_improvement,wins
0,0.50,0.46,0.50,0.15,0.05,0.05,0.75,True,0.146330,0.000056,0.000023,3
1,0.50,0.46,0.50,0.15,0.10,0.05,0.70,True,0.146349,0.000037,0.000007,3
2,0.52,0.46,0.50,0.15,0.10,0.05,0.70,True,0.146356,0.000030,-0.000003,2
3,0.48,0.46,0.50,0.15,0.05,0.05,0.75,True,0.146324,0.000061,-0.000003,2
4,0.50,0.48,0.50,0.15,0.10,0.05,0.70,True,0.146346,0.000040,-0.000007,2
5,0.50,0.48,0.50,0.15,0.05,0.05,0.75,True,0.146359,0.000027,-0.000007,2
6,0.48,0.50,0.50,0.15,0.10,0.05,0.70,True,0.146354,0.000031,-0.000007,2
7,0.50,0.46,0.52,0.15,0.05,0.05,0.75,True,0.146363,0.000022,-0.000013,2
8,0.48,0.50,0.50,0.15,0.05,0.05,0.75,True,0.146359,0.000027,-0.000033,2
9,0.50,0.50,0.50,0.15,0.10,0.05,0.70,True,0.146350,0.000036,-0.000033,2


## 7. 전체 Train으로 2D·3D·4D Test 예측

In [16]:
pred_tree=tree_predict(X,X_test,y)
materials=neighbor_materials(X,X_test,y)
pred_pair=np.quantile(materials[2],float(selected.pair_quantile),axis=1)
pred_triple=np.quantile(materials[3],float(selected.triple_quantile),axis=1)
pred_quad=np.quantile(materials[4],float(selected.quad_quantile),axis=1)
pred_v20=(
    float(selected.tree_weight)*pred_tree
    +float(selected.pair_weight)*pred_pair
    +float(selected.triple_weight)*pred_triple
    +float(selected.quad_weight)*pred_quad
)
if bool(selected.rounding): pred_v20=np.round(pred_v20,2)
pred_v20=np.clip(pred_v20,0,1)
print("Prediction range:",pred_v20.min(),pred_v20.max())
print("첫 10개:",pred_v20[:10])

Prediction range: 0.0 1.0
첫 10개: [0.56 0.97 0.19 0.49 0.53 0.46 0.78 0.46 0.57 0.88]


## 8. 제출 파일과 원본 기준 재현 확인

In [17]:
submission_v21=submission.copy()
submission_v21[ID_COLUMN]=test[ID_COLUMN]
submission_v21[TARGET]=pred_v20
output_path=OUTPUT_DIR/"submit_v21_precision_fixed_neighbor.csv"
submission_v21.to_csv(output_path,index=False,encoding="utf-8")

# 팀 원본 0.1272333 모델의 Test 예측도 별도 보관
baseline_test=np.clip(np.round(
    .85*pred_tree+.15*np.quantile(materials[2],.52,axis=1),2
),0,1)
baseline_path=OUTPUT_DIR/"submit_v21_reconstructed_team_best.csv"
baseline_frame=submission.copy(); baseline_frame[TARGET]=baseline_test
baseline_frame.to_csv(baseline_path,index=False,encoding="utf-8")

print("Saved candidate:",output_path)
print("Saved reconstructed baseline:",baseline_path)
print("Candidate vs baseline mean abs change:",np.abs(pred_v20-baseline_test).mean())
submission_v21.head(10)

Saved candidate: /Users/bitsaem/Desktop/stress_project/submit_v21_precision_fixed_neighbor.csv
Saved reconstructed baseline: /Users/bitsaem/Desktop/stress_project/submit_v21_reconstructed_team_best.csv
Candidate vs baseline mean abs change: 0.004893333333333334


,ID,stress_score
0,TEST_0000,0.56
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.49
4,TEST_0004,0.53
5,TEST_0005,0.46
6,TEST_0006,0.78
7,TEST_0007,0.46
8,TEST_0008,0.57
9,TEST_0009,0.88


## 제출 기준

- 먼저 `submit_v21_reconstructed_team_best.csv`가 팀 원본 제출과 일치하는지 확인
- `Wins: 3 / 3`
- `Worst split improvement >= 0.0001`
- `Selected 4D weight > 0`
- 모든 조건을 만족할 때만 V21 제출 후보로 검토